# 🏥 Linear Regression Model for Maternal Mortality Prediction

Complete implementation and analysis of Linear Regression models for predicting Maternal Mortality Ratios across countries.

## Section 1️⃣: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from scipy import stats
from scipy.stats import pearsonr
import warnings
warnings.filterwarnings('ignore')

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

## Section 2️⃣: Load Dataset

In [ ]:
DATA_PATH = "/home/chacha/Desktop/MACHINE LEARNING/Maternity_Mortality_Prediction_Model-main/Maternal_Mortality.csv"
df = pd.read_csv(DATA_PATH)

print(f"Dataset Shape: {df.shape}")
print(f"\nFirst rows:")
print(df.head())
print(f"\nDescriptive Statistics:")
print(df.describe().round(2))

## Section 3️⃣: Data Preprocessing

In [ ]:
mmr_columns = [col for col in df.columns if 'Maternal Mortality' in col]
X_cols = mmr_columns[:-1]
y_col = mmr_columns[-1]

X = df[X_cols].copy()
y = df[y_col].copy()

if 'HDI Rank (2021)' in df.columns:
    X_hdi = df[['HDI Rank (2021)']].copy()
    X_hdi = X_hdi.fillna(X_hdi.mean())
    X = pd.concat([X, X_hdi], axis=1)

valid_mask = ~y.isnull()
X = X[valid_mask].reset_index(drop=True)
y = y[valid_mask].reset_index(drop=True)

print(f"Features: {X.shape[1]} variables, {len(X)} countries")
print(f"Target: Mean={y.mean():.2f}, SD={y.std():.2f}, Range=[{y.min():.0f}, {y.max():.0f}]")

### Target Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(y, bins=30, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].axvline(y.mean(), color='red', linestyle='--', lw=2, label=f'Mean: {y.mean():.1f}')
axes[0].set_xlabel('MMR 2021')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

axes[1].boxplot(y, vert=True, patch_artist=True, boxprops=dict(facecolor='lightblue', alpha=0.7),
                medianprops=dict(color='red', lw=2))
axes[1].set_ylabel('MMR 2021')
axes[1].set_title('Box Plot')
axes[1].grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

### Correlation Analysis

In [ ]:
sample_X = X[X_cols[-5:]].copy()
sample_X['Target_2021'] = y
fig, ax = plt.subplots(figsize=(8, 6))
corr_matrix = sample_X.corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True, linewidths=1, ax=ax)
ax.set_title('Correlation Heatmap (Recent Years + Target)')
plt.tight_layout()
plt.show()

## Section 4️⃣: Simple Linear Regression

In [ ]:
X_simple = X[[X_cols[-1]]].copy()
X_train_simple, X_test_simple, y_train_simple, y_test_simple = train_test_split(
    X_simple, y, test_size=0.2, random_state=42
)

lr_simple = LinearRegression()
lr_simple.fit(X_train_simple, y_train_simple)

print(f"Simple LR: y = {lr_simple.intercept_:.4f} + {lr_simple.coef_[0]:.4f} × MMR_2020")
y_pred_train_simple = lr_simple.predict(X_train_simple)
y_pred_test_simple = lr_simple.predict(X_test_simple)

### Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(X_train_simple, y_train_simple, alpha=0.6, s=50, label='Training')
X_line = np.array([[X_train_simple.min()[0]], [X_train_simple.max()[0]]])
axes[0].plot(X_line, lr_simple.predict(X_line), 'r--', lw=2)
axes[0].set_xlabel('MMR 2020')
axes[0].set_ylabel('MMR 2021')
axes[0].set_title('Training Data')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].scatter(X_test_simple, y_test_simple, alpha=0.6, s=50, label='Testing', color='green')
axes[1].plot(X_line, lr_simple.predict(X_line), 'r--', lw=2)
axes[1].set_xlabel('MMR 2020')
axes[1].set_ylabel('MMR 2021')
axes[1].set_title('Testing Data')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Section 5️⃣: Multiple Linear Regression

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr_multiple = LinearRegression()
lr_multiple.fit(X_train_scaled, y_train)

print(f"Multiple LR: {X_train.shape[1]} features, Intercept={lr_multiple.intercept_:.4f}")
print(f"Train: {len(X_train)}, Test: {len(X_test)}")

feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': lr_multiple.coef_,
    'Abs_Coefficient': np.abs(lr_multiple.coef_)
}).sort_values('Abs_Coefficient', ascending=False)

print(f"\nTop 10 Features:")
print(feature_importance.head(10)[['Feature', 'Coefficient']].to_string(index=False))

### Feature Importance

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
top_features = feature_importance.head(10)
colors = ['green' if x > 0 else 'red' for x in top_features['Coefficient']]
ax.barh(range(len(top_features)), top_features['Coefficient'], color=colors, alpha=0.7, edgecolor='black')
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features['Feature'])
ax.set_xlabel('Coefficient')
ax.set_title('Top 10 Feature Coefficients')
ax.grid(True, alpha=0.3, axis='x')
ax.axvline(0, color='black', linestyle='-', lw=0.8)
plt.tight_layout()
plt.show()

y_pred_train = lr_multiple.predict(X_train_scaled)
y_pred_test = lr_multiple.predict(X_test_scaled)

## Section 6️⃣: Model Evaluation

In [ ]:
simple_r2_train = r2_score(y_train_simple, y_pred_train_simple)
simple_r2_test = r2_score(y_test_simple, y_pred_test_simple)
simple_rmse_test = np.sqrt(mean_squared_error(y_test_simple, y_pred_test_simple))
simple_mae_test = mean_absolute_error(y_test_simple, y_pred_test_simple)

train_r2 = r2_score(y_train, y_pred_train)
train_mse = mean_squared_error(y_train, y_pred_train)
train_rmse = np.sqrt(train_mse)
train_mae = mean_absolute_error(y_train, y_pred_train)

test_r2 = r2_score(y_test, y_pred_test)
test_mse = mean_squared_error(y_test, y_pred_test)
test_rmse = np.sqrt(test_mse)
test_mae = mean_absolute_error(y_test, y_pred_test)

mult_r2_train = train_r2
mult_r2_test = test_r2
mult_rmse_test = test_rmse
mult_mae_test = test_mae

print(f"Training: R²={train_r2:.6f}, RMSE={train_rmse:.6f}")
print(f"Testing:  R²={test_r2:.6f}, RMSE={test_rmse:.6f}")
print(f"\nComparison:")
print(f"Simple LR:   R²={simple_r2_test:.6f}, RMSE={simple_rmse_test:.6f}")
print(f"Multiple LR: R²={test_r2:.6f}, RMSE={test_rmse:.6f}")

### Metrics Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Performance Metrics', fontsize=14)

metrics_names = ['Train', 'Test']
axes[0, 0].bar(metrics_names, [train_r2, test_r2], color=['blue', 'orange'], alpha=0.7)
axes[0, 0].set_ylabel('R² Score')
axes[0, 0].set_ylim([0, 1])
axes[0, 0].grid(True, alpha=0.3, axis='y')

axes[0, 1].bar(metrics_names, [train_mse, test_mse], color=['blue', 'orange'], alpha=0.7)
axes[0, 1].set_ylabel('MSE')
axes[0, 1].grid(True, alpha=0.3, axis='y')

axes[1, 0].bar(metrics_names, [train_rmse, test_rmse], color=['blue', 'orange'], alpha=0.7)
axes[1, 0].set_ylabel('RMSE')
axes[1, 0].grid(True, alpha=0.3, axis='y')

axes[1, 1].bar(metrics_names, [train_mae, test_mae], color=['blue', 'orange'], alpha=0.7)
axes[1, 1].set_ylabel('MAE')
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## Section 7️⃣: Predictions & Analysis

In [ ]:
print(f"Sample Predictions (First 10):")
pred_df = pd.DataFrame({
    'Actual': y_test.values[:10],
    'Predicted': y_pred_test[:10],
    'Error': y_test.values[:10] - y_pred_test[:10]
})
print(pred_df.to_string(index=True))

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

min_val = min(y_test.min(), y_pred_test.min())
max_val = max(y_test.max(), y_pred_test.max())
axes[0].scatter(y_test, y_pred_test, alpha=0.6, s=60, edgecolors='black', linewidth=0.5)
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', lw=2)
axes[0].set_xlabel('Actual')
axes[0].set_ylabel('Predicted')
axes[0].set_title(f'Test (R²={test_r2:.4f})')
axes[0].grid(True, alpha=0.3)

min_val = min(y_train.min(), y_pred_train.min())
max_val = max(y_train.max(), y_pred_train.max())
axes[1].scatter(y_train, y_pred_train, alpha=0.6, s=60, edgecolors='black', linewidth=0.5, color='green')
axes[1].plot([min_val, max_val], [min_val, max_val], 'r--', lw=2)
axes[1].set_xlabel('Actual')
axes[1].set_ylabel('Predicted')
axes[1].set_title(f'Training (R²={train_r2:.4f})')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

errors = y_test - y_pred_test
print(f"\nError Stats: Mean={errors.mean():.4f}, Std={errors.std():.4f}")

## Section 8️⃣: Residual Analysis

In [ ]:
residuals_test = y_test - y_pred_test

print(f"Residuals: Mean={residuals_test.mean():.6f}, Std={residuals_test.std():.6f}")

if len(residuals_test) <= 5000:
    stat, pvalue = stats.shapiro(residuals_test)
    print(f"Shapiro-Wilk p-value: {pvalue:.6f}")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Residual Analysis', fontsize=14)

axes[0, 0].hist(residuals_test, bins=20, color='steelblue', edgecolor='black', alpha=0.7)
axes[0, 0].axvline(0, color='red', linestyle='--', lw=2)
axes[0, 0].set_xlabel('Residuals')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].grid(True, alpha=0.3, axis='y')

axes[0, 1].scatter(y_pred_test, residuals_test, alpha=0.6, s=50, edgecolors='black', linewidth=0.5)
axes[0, 1].axhline(0, color='red', linestyle='--', lw=2)
axes[0, 1].set_xlabel('Fitted')
axes[0, 1].set_ylabel('Residuals')
axes[0, 1].grid(True, alpha=0.3)

stats.probplot(residuals_test, dist="norm", plot=axes[1, 0])
axes[1, 0].set_title('Q-Q Plot')
axes[1, 0].grid(True, alpha=0.3)

std_residuals = residuals_test / residuals_test.std()
axes[1, 1].plot(range(len(std_residuals)), std_residuals, 'o-', alpha=0.6, markersize=4)
axes[1, 1].axhline(0, color='red', linestyle='--', lw=2)
axes[1, 1].axhline(2, color='orange', linestyle='--', lw=1, alpha=0.7, label='±2 SD')
axes[1, 1].axhline(-2, color='orange', linestyle='--', lw=1, alpha=0.7)
axes[1, 1].set_xlabel('Order')
axes[1, 1].set_ylabel('Std Residuals')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Section 9️⃣: Summary

In [ ]:
print("\nMODEL SUMMARY")
print(f"Features: {X_train.shape[1]}, Countries: {len(X)}, Train/Test: {len(y_train)}/{len(y_test)}")
print(f"\nPerformance:")
print(f"Simple LR:     R²={simple_r2_test:.4f}, RMSE={simple_rmse_test:.4f}")
print(f"Multiple LR:   R²={test_r2:.4f}, RMSE={test_rmse:.4f}")
print(f"\nTop Features:")
print(feature_importance.head(5)[['Feature', 'Coefficient']].to_string(index=False))
print(f"\nData Summary:")
print(f"  Countries: {len(df)}")
print(f"  MMR Range: {y.min():.0f}-{y.max():.0f} per 100k")
print(f"  Years: 31 (1990-2020)")
print(f"  Generated: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}")